# 03 Modelado y evaluacion

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'scripts'))
from data_pipeline import audit


Experimentos reproducibles: ejecutar `python scripts/experiments.py --evaluate` desde la raíz. Este notebook revisa los artefactos ya calculados, verifica sus métricas y no vuelve a ajustar después de observar prueba. GridSearchCV, selección y congelación están implementados en scripts/experiments.py.

In [ ]:
frozen=json.loads((ROOT/'reports/seleccion_congelada.json').read_text())
display(pd.Series(frozen['selected']))
display(pd.read_csv(ROOT/'reports/comparacion_modelos.csv'))

In [ ]:
from verify import verify
verify()
display(pd.read_csv(ROOT/'reports/comparacion_ventanas.csv'))
display(pd.read_csv(ROOT/'reports/metricas_folds.csv'))

In [ ]:
display(Image(filename=str(ROOT/'reports/figures/aprendizaje.png')))

Promedios de tres folds con prefijos de fechas completas. El eje horizontal es el tamaño medio de entrenamiento. Son diagnósticos de desarrollo, no una estimación independiente del rendimiento seleccionado.

In [ ]:
display(Image(filename=str(ROOT/'reports/figures/confusion.png')))

Filas normalizadas por clase real. El modelo seleccionado reconoce el 35,1 % de positivos y el 82,8 % de negativos; la sensibilidad a positivos es limitada.

In [ ]:
display(Image(filename=str(ROOT/'reports/figures/roc.png')))

La ROC compara sensibilidad y falsas alarmas al variar el umbral. No se utilizó para cambiar el umbral fijado en 0,5.

In [ ]:
display(Image(filename=str(ROOT/'reports/figures/precision_recall.png')))

La curva precisión–recall centra la atención en positivos. La línea horizontal es la prevalencia en prueba (37/66), no un modelo adicional entrenado.

## Conclusiones

La selección por validación fue RF_7d_base, con F1 medio 0.653 y desviación entre bloques 0.097.

En prueba, su F1 fue 0.473, precisión 0.722 y recall 0.351. Detectó 13 de los 37 positivos y omitió 24. Clasificó correctamente 24 de los 29 negativos.

Ampliar a 14 días no mostró una mejora consistente: el F1 de validación bajó en tres de cuatro comparaciones equivalentes. En prueba mejoró solo LR sin IITH. Son diferencias descriptivas, sin evidencia de significancia estadística.

RF con 7 días e IITH logró el mayor F1 de prueba (0,481), pero no reemplaza al modelo seleccionado previamente. RF con 14 días e IITH logró el mayor ROC-AUC (0,753): el ordenamiento de probabilidades y el F1 con umbral 0,5 miden propiedades diferentes.

Agregar IITH no mejoró F1 de validación en ninguna comparación equivalente. Su utilidad futura no queda descartada por este conjunto pequeño.

La caída entre validación y prueba señala generalización temporal limitada. Las curvas de aprendizaje muestran diferencias entre ajuste y validación, compatibles con sobreajuste y cambios de distribución. No permiten separar sus causas.